# Transformer

## Transformer 架构

figs/transformer.png


## Transformer 训练
figs/transformer2.png, 该图片来源于网络

## Transformer 推理过程

下面是 transformer 的代码实现，不包括 codebook 的预训练部分，仅有 transformer 生成模型

In [ ]:
# multi-head attention implementation
import torch
import torch.nn as nn

# basic moudle for multi-head attention, feedforward
class MultiHeadAttentionWithMask(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.W_qkv = nn.Linear(hidden_dim, hidden_dim * 3)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        # x: [batch_size, seq_len, hidden_dim]
        qkv = self.W_qkv(x)  # [batch_size, seq_len, hidden_dim * 3]
        q, k, v = qkv.chunk(3, dim=-1)  # each: [batch_size, seq_len, hidden_dim]

        # [batch_size, seq_len, hidden_dim] -> [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask can be [seq_len, seq_len], [batch_size, seq_len, seq_len],
            # or [batch_size, 1, seq_len, seq_len]. It is broadcast across heads.
            if mask.dim() == 2:
                mask = mask.unsqueeze(1)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            elif mask.dim() != 4:
                raise ValueError("mask must have shape [T,T], [B,T,T], or [B,1,T,T]")
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)  # [batch_size, num_heads, seq_len, seq_len]

        # [batch_size, num_heads, seq_len, head_dim]
        output_heads = torch.matmul(attention_weights, v)

        # [batch_size, num_heads, seq_len, head_dim] -> [batch_size, seq_len, hidden_dim]
        output_concat = output_heads.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        output = self.W_o(output_concat)

        return output, attention_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        # self.W_qkv = nn.Linear(hidden_dim, hidden_dim * 3)
        self.W_o = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, q, k, v, mask=None):
        batch_size, seq_len, _ = q.size()

        # q, k, v: [batch_size, seq_len, hidden_dim]
        # For self-attention, q, k, v are all the same
        # For encoder-decoder attention, q comes from decoder, k and v come from encoder

        # [batch_size, seq_len, hidden_dim] -> [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # [batch_size, seq_len, hidden_dim] -> [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # mask can be [seq_len, seq_len], [batch_size, seq_len, seq_len],
            # or [batch_size, 1, seq_len, seq_len]. It is broadcast across heads.
            if mask.dim() == 2:
                mask = mask.unsqueeze(1)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            elif mask.dim() != 4:
                raise ValueError("mask must have shape [T,T], [B,T,T], or [B,1,T,T]")
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)  # [batch_size, num_heads, seq_len, seq_len]

        # [batch_size, num_heads, seq_len, head_dim]
        output_heads = torch.matmul(attention_weights, v)

        # [batch_size, num_heads, seq_len, head_dim] -> [batch_size, seq_len, hidden_dim]
        output_concat = output_heads.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        output = self.W_o(output_concat)

        return output, attention_weights

class FeedForward(nn.Module):
    def __init__(self, hidden_dim, ff_dim):
        super().__init__()
        self.linear1 = nn.Linear(hidden_dim, ff_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(ff_dim, hidden_dim)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

In [ ]:

# Positional Encoding implementation
class PositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, hidden_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / hidden_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, hidden_dim]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
# Transformer Encoder Layer implementation
class TransformerEncoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(hidden_dim, num_heads)
        self.ffn = FeedForward(hidden_dim, ff_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [batch_size, seq_len, hidden_dim]
        # Multi-head attention
        attn_output, _ = self.mha(x, mask)
        # add & norm
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)

        # Feedforward network
        ffn_output = self.ffn(x)
        # add & norm
        x = x + self.dropout2(ffn_output)
        x = self.norm2(x)

        return x

# Transformer Encoder implementation
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)
    
# Transformer Decoder Layer implementation
class TransformerDecoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttentionWithMask(hidden_dim, num_heads)
        self.mha2 = MultiHeadAttention(hidden_dim, num_heads)
        self.ffn = FeedForward(hidden_dim, ff_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
     

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # x: [batch_size, tgt_seq_len, hidden_dim]
        # enc_output: [batch_size, src_seq_len, hidden_dim]
        
        # Masked multi-head attention (self-attention)
        attn_output1, _ = self.mha1(x, tgt_mask)
        # add & norm
        x = x + self.dropout1(attn_output1)
        x = self.norm1(x)

        # Multi-head attention (encoder-decoder attention)
        attn_output2, _ = self.mha2(q=x, k=enc_output, v=enc_output, mask=src_mask)
        # add & norm
        x = x + self.dropout2(attn_output2)
        x = self.norm2(x)

        # Feedforward network
        ffn_output = self.ffn(x)
        # add & norm
        x = x + self.dropout3(ffn_output)
        x = self.norm3(x)

        return x
    
# Transformer Decoder implementation
class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return self.norm(x)

# Transformer model implementation
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.src_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.tgt_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.positional_encoding = PositionalEncoding(hidden_dim)
        self.encoder = TransformerEncoder(num_layers, hidden_dim, num_heads, ff_dim, dropout)
        self.decoder = TransformerDecoder(num_layers, hidden_dim, num_heads, ff_dim, dropout)
        self.output_linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src_input, tgt_input, src_mask=None, tgt_mask=None):
        # src_input: [batch_size, src_seq_len]
        # tgt_input: [batch_size, tgt_seq_len]
        
        # Embedding and positional encoding
        src_embedded = self.positional_encoding(self.src_embedding(src_input))
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt_input))

        # Encoder
        enc_output = self.encoder(src_embedded, src_mask)

        # Decoder
        dec_output = self.decoder(tgt_embedded, enc_output, src_mask, tgt_mask)

        # Output linear layer
        output = self.output_linear(dec_output)  # [batch_size, tgt_seq_len, tgt_vocab_size]
        output = torch.softmax(output, dim=-1)  # Apply softmax to compute logits

        return output

    def compute_loss(self, src_input, tgt_input, tgt_output, src_mask=None, tgt_mask=None):
        # Forward pass
        output = self.forward(src_input, tgt_input, src_mask, tgt_mask)
        
        # Compute loss
        loss_fn = nn.CrossEntropyLoss(ignore_index=0)  # Assuming 0 is the padding index
        loss = loss_fn(output.view(-1, output.size(-1)), tgt_output.view(-1))
        
        return loss

    def generate(self, src_input, max_len, src_mask=None):
        # src_input: [batch_size, src_seq_len]
        batch_size = src_input.size(0)
        generated_seq = torch.zeros(batch_size, max_len, dtype=torch.long).to(src_input.device)
        generated_seq[:, 0] = 1  # Assuming 1 is the start token index

        for t in range(1, max_len):
            tgt_input = generated_seq[:, :t]  # [batch_size, t]
            output = self.forward(src_input, tgt_input, src_mask)  # [batch_size, t, vocab_size]
            next_token = output[:, -1, :].argmax(dim=-1)  # [batch_size]
            generated_seq[:, t] = next_token

            if (next_token == 2).all():  # Assuming 2 is the end token index
                break

        return generated_seq